In [1]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

In [2]:
options = Options()
options.add_argument("--headless=new")
driver = webdriver.Chrome(options=options)
driver.get('https://www.wildflower.org/plants/result.php?id_plant=enpe4')
time.sleep(2)
page_source = driver.page_source
driver.quit()

In [3]:
soup = BeautifulSoup(page_source, 'html.parser')

# this is the scientific name of the plant
titles = soup.find_all('h2', class_='tax_sn')
for title in titles:
    print(title.text.strip())

Engelmannia peristenia


In [9]:
# Scrape links from NPSOT website

npsot_unnecessary_link_words = [
    'login',
    'signup',
    'smugmug',
    'facebook',
    'twitter',
    'instagram',
    'linkedin',
    'youtube',
    'chapters',
    'people',
    'contact',
]

url = 'https://npsot.org/'

driver = webdriver.Chrome(options=options)
driver.get(url)
time.sleep(2)
page_source = driver.page_source
driver.quit()
soup = BeautifulSoup(page_source, 'html.parser')
all_links = []
links = soup.find_all('a')
for link in links:
    href = link.get('href')
    if href and href.startswith('http') and 'login' not in href and 'signup' not in href:
        all_links.append(href)

# remove duplicates
all_links = list(set([link.strip() for link in all_links]))

# remove unnecessary
final_links = []
for link in all_links:
    if not any(word in link for word in npsot_unnecessary_link_words):
        final_links.append(link)


print(len(all_links), len(final_links))

154 106


In [ ]:
# Save the links as markdown file
with open('npsot_links.md', 'w') as f:
    for link in final_links:
        f.write(f'- {link}\n')


In [16]:
# save webpage as a markdown file. This is to test if we can use the markdown file to create a vector store and then use it for question answering.
with open('npsot_webpages.md', 'w') as f:
    for i, link in enumerate(final_links):
        driver = webdriver.Chrome(options=options)
        driver.get(link)
        time.sleep(2)
        page_source = driver.page_source
        driver.quit()
        soup = BeautifulSoup(page_source, 'html.parser')
        titles = soup.find_all('h1')
        if len(titles) > 0:
            filename = titles[0].text.strip().replace(' ', '_').replace('-', '_') + '.md'
        print(filename)
        # write soup content to markdown file

        with open(filename, 'w') as f:
            text = soup.get_text()
            # remove multiple newlines
            text = '\n'.join(
                [line.strip() for line in text.splitlines() if line.strip() != '']
            )
            f.write(text)
        if i >= 2:
            break

Native_Landscape_Certification_Program.md
Annual_Symposia.md
Get_Started!.md


In [17]:
link = "https://www.npsot.org/posts/native-plant/zinnia-anomala/"
driver = webdriver.Chrome(options=options)
driver.get(link)
time.sleep(2)
page_source = driver.page_source
driver.quit()
soup = BeautifulSoup(page_source, 'html.parser')
titles = soup.find_all('h1')
if len(titles) > 0:
    filename = titles[0].text.strip().replace(' ', '_').replace('-', '_') + '.md'
print(filename)
# write soup content to markdown file

with open(filename, 'w') as f:
    text = soup.get_text()
    # remove multiple newlines
    text = '\n'.join(
        [line.strip() for line in text.splitlines() if line.strip() != '']
    )
    f.write(text)



Shortray_Zinnia.md


In [34]:
# find text in div class elementor-widget-container
reference_sources = [
    'wikipedia',
    'wildflower',
    'bonap',
]

divs = soup.find_all('div', class_='elementor-widget-container')

# find all the links in div class elementor-shortcode
shortcode_divs = soup.find_all('div', class_='elementor-shortcode')
reference_links = []
for div in shortcode_divs:
    links = div.find_all('a')
    for link in links:
        href = link.get('href')
        if href and href.startswith('http') and 'leaflet' not in href:
            reference_links.append(href)
reference_links = list(set(reference_links))
reference_links

['https://www.npsot.org/wp-content/uploads/2026/03/Zinnia-anomala-full-view.avif',
 'https://itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=38713#null',
 'https://bonap.net/TDC/Image/Map?taxonType=Species&taxonId=4817&locationType=County&mapType=Normal',
 'https://portal.torcherbaria.org/portal/taxa/index.php?taxon=Zinnia+anomala&formsubmit=Search+Terms',
 'https://www.wildflower.org/plants/result.php?id_plant=ZIAN',
 'https://en.wikipedia.org/wiki/Zinnia_anomala#:~:text=Zinnia%20anomala%20is%20a%20profusely%20branched%20subshrub,florets%2C%20other%20times%20with%205%2D8%20yellow%20rays.',
 'https://plants.ces.ncsu.edu/plants/zinnia-anomala/']

In [ ]:
# find all the link from this NPSOT database page
# https://www.npsot.org/resources/native-plants/native-plants-database/

# Get all the plant families from the respective pages
# Use those to get to the wildflower website and search for the species

# Scrape the rest of the websties from NPSOT references section

https://www.npsot.org/wp-content/uploads/2026/03/Zinnia-anomala-full-view.avif
https://itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=38713#null
https://bonap.net/TDC/Image/Map?taxonType=Species&taxonId=4817&locationType=County&mapType=Normal
https://portal.torcherbaria.org/portal/taxa/index.php?taxon=Zinnia+anomala&formsubmit=Search+Terms
https://leafletjs.com
https://www.wildflower.org/plants/result.php?id_plant=ZIAN
https://en.wikipedia.org/wiki/Zinnia_anomala#:~:text=Zinnia%20anomala%20is%20a%20profusely%20branched%20subshrub,florets%2C%20other%20times%20with%205%2D8%20yellow%20rays.
https://plants.ces.ncsu.edu/plants/zinnia-anomala/


In [ ]:
# bonap is just a map - find a way to save the image from the page
#